# Intent Lab: классификация обращений и отказ от неуверенных решений

**Вопрос:** какую долю обращений можно автоматически распределить при целевой точности 95%?

Данные: английский BANKING77, 77 банковских намерений, CC BY 4.0.
Источник: https://github.com/PolyAI-LDN/task-specific-datasets
Авторы: Casanueva et al., *Efficient Intent Detection with Dual Sentence Encoders*, 2020.

Этот ноутбук содержит выполненный анализ сохранённых результатов. Полное обучение:
`python experiment.py --model all` из корня проекта. Здесь не используются API генеративных LLM.


In [1]:
import json
from itertools import combinations
from pathlib import Path

import pandas as pd


In [2]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
REPORTS = ROOT / "reports"
audit = json.loads((REPORTS / "data_audit.json").read_text(encoding="utf-8"))
print(json.dumps(audit, ensure_ascii=False, indent=2))


{
  "dataset": "BANKING77",
  "revision": "57ec275d8078af65b7731c2a98be812d844a6d6b",
  "license": "CC-BY-4.0",
  "language": "English",
  "source": {
    "train": {
      "url": "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/train.csv",
      "sha256": "b06e26ac675513959a63135f11b94ea7786ed02da65db93a5650d8838cbc664b",
      "rows": 10003
    },
    "test": {
      "url": "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/test.csv",
      "sha256": "d12d6e3bc4c3103966ae786dc435913c0c563dfa328f5a3646d0e62cfeeb474d",
      "rows": 3080
    }
  },
  "splits": {
    "train": 6962,
    "validation": 1492,
    "calibration": 1492,
    "test": 3080
  },
  "classes": 77,
  "removed_from_train": 57,
  "conflicting_train_keys": 1,
  "test_duplicate_keys": 4,
  "normalization": "NFKC, casefold, punctuation/whitespace normalization",
  "limitations": 

## 1. Проверяем данные и отсутствие утечки

Сначала из обучающей части удалены совпадения с официальным тестом, повторы и
тексты с конфликтующей разметкой. Сравнение нормализует регистр и пунктуацию.
Семантические перефразировки автоматически не удаляются.
Train/validation/calibration разделены стратифицированно с seed=42.
Test сохранён в исходном составе. Параметры выбираются по validation. Тест уже использовался в первом эксперименте: повторная оценка не является новой слепой проверкой.


In [3]:
frames = {name: pd.read_csv(ROOT / "data" / f"{name}.csv") for name in ["train", "validation", "calibration", "test"]}
for a, b in combinations(frames, 2):
    assert set(frames[a]["key"]).isdisjoint(frames[b]["key"])
print("Пересечений нормализованных текстов между выборками нет.")
counts = pd.read_csv(REPORTS / "class_counts.csv")
print("Количество обучающих примеров на класс:")
print(counts["train"].describe().to_string())
print("\nСамые малочисленные категории:")
print(counts.sort_values("train").head(10).to_string(index=False))


Пересечений нормализованных текстов между выборками нет.
Количество обучающих примеров на класс:
count     77.000000
mean      90.415584
std       23.042495
min       25.000000
25%       78.000000
50%       88.000000
75%      111.000000
max      131.000000

Самые малочисленные категории:
                      label  calibration  test  train  validation
    contactless_not_working            5    40     25           5
   virtual_card_not_working            6    40     29           6
            card_acceptance            8    40     40           9
             card_swallowed            9    40     43           9
        lost_or_stolen_card           12    40     57          13
           compromised_card           13    40     60          12
                atm_support           13    40     60          12
              top_up_limits           13    40     63          14
get_disposable_virtual_card           15    40     67          14
            receiving_money           14    40     

## 2. Обучаемые модели

**TF-IDF + Logistic Regression:** словесные 1-2-граммы и символьные 3-5-граммы.
Векторизатор и классификатор обучены только на train. C выбирается по validation macro-F1.

**BERT Tiny:** предобученный двухслойный трансформер, hidden size 128. Все веса
дообучаются вместе с новой головой на 77 классов. AdamW, lr BERT из {1e-4, 3e-4}, lr головы 1e-3, batch=32,
максимум 64 токена. До 35 эпох на вариант; warmup 10% и линейное снижение скорости. Ранняя остановка: patience=5, min_delta=0,001. Лучший checkpoint выбирается по validation macro-F1.
TF-IDF не обрезает длинные обращения; это различие следует учитывать.

Предобученная модель: https://huggingface.co/prajjwal1/bert-tiny


In [4]:
reports = {name: json.loads((REPORTS / f"{name}.json").read_text(encoding="utf-8")) for name in ["tfidf", "bert_tiny"]}
print("Подбор C:")
print(pd.DataFrame(reports["tfidf"]["experiments"]).to_string(index=False))
print("\nОбучение BERT:")
print(pd.DataFrame(reports["bert_tiny"]["history"]).to_string(index=False))
print("Лучшая эпоха:", reports["bert_tiny"]["best_epoch"])

print("Выбранная скорость:", reports["bert_tiny"]["learning_rate"])
for trial in reports["bert_tiny"]["trials"]:
    print("lr:", trial["learning_rate"], "best validation macro-F1:", max(e["validation_macro_f1"] for e in trial["history"]))


Подбор C:
   C  validation_macro_f1
 1.0             0.889536
 4.0             0.898726
16.0             0.902759

Обучение BERT:
 epoch  train_loss  validation_macro_f1
     1    4.318157             0.025492
     2    3.890212             0.241879
     3    2.787390             0.456893
     4    1.876198             0.637293
     5    1.356162             0.746189
     6    1.033201             0.780364
     7    0.818625             0.820169
     8    0.660648             0.829378
     9    0.558424             0.850101
    10    0.469543             0.865109
    11    0.410352             0.876551
    12    0.358389             0.879555
    13    0.331029             0.881605
    14    0.284621             0.885065
    15    0.261643             0.881951
    16    0.241217             0.883909
    17    0.216257             0.885124
    18    0.206449             0.883518
    19    0.187210             0.886770
    20    0.166193             0.891820
    21    0.159689            

## 3. Выбираем порог на calibration, оцениваем на test

Порог максимизирует долю автоматически принятых сообщений при наблюдаемой
точности ≥95% на calibration и минимум 50 принятых сообщениях. Если такого
порога нет, все сообщения направляются человеку. Равные оценки принимаются вместе.

**Это не гарантия 95% на новых данных.** Малые группы имеют большую статистическую
неопределённость. Оценки модели не были откалиброваны temperature scaling.


In [5]:
rows = []
for name, r in reports.items():
    rows.append({"model": name, "macro_f1": r["test_macro_f1"], "accuracy": r["test_accuracy"],
                 "threshold": r["threshold"], "coverage": r["test"]["coverage"],
                 "accepted_accuracy": r["test"]["accepted_accuracy"], "accepted": r["test"]["accepted"]})
print(pd.DataFrame(rows).to_string(index=False))


    model  macro_f1  accuracy  threshold  coverage  accepted_accuracy  accepted
    tfidf  0.901376  0.901299   0.482879  0.877273           0.956329      2702
bert_tiny  0.881460  0.881494   0.802176  0.855195           0.949127      2634


## 4. Разбираем ошибки

Проверяем уверенные ошибки, а не только среднюю метрику. Для каждого примера:
похожа ли исходная метка на правильную? Есть ли неоднозначность? Не перепутаны ли
близкие банковские операции? Ответы после анализа теста - гипотезы для следующего
эксперимента, которому понадобится новая независимая проверка.


In [6]:
for name in reports:
    errors = pd.read_csv(REPORTS / f"{name}_errors.csv")
    print("\nМодель:", name, "Ошибок:", len(errors))
    print(errors.head(8).to_string(index=False))
    print("\nЧаще всего путаются:")
    print(errors.groupby(["label", "predicted"]).size().sort_values(ascending=False).head(8).to_string())



Модель: tfidf Ошибок: 304
                                                        text                                   label                               predicted  confidence  accepted
                                     My transfer is pending. balance_not_updated_after_bank_transfer                        pending_transfer    0.993151      True
               What can I use a virtual disposable card for?                  disposable_card_limits             get_disposable_virtual_card    0.992897      True
     how many transactions can i make with a disposable card             get_disposable_virtual_card                  disposable_card_limits    0.992634      True
how long dies it take for transfers to reflect on my balance                         transfer_timing balance_not_updated_after_bank_transfer    0.989485      True
                              can I have a non virtual card?                     order_physical_card                    getting_virtual_card    0.974469      

## 5. Что можно утверждать по результатам

Это измерения на одном английском датасете, а не результаты внедрения.
Неизвестные намерения и запросы вне банковской тематики могут получать высокую
уверенность; порог не заменяет отдельную проверку out-of-domain.
Проект демонстрирует подготовку данных, supervised learning, fine-tuning,
выбор модели по validation и selective classification.

Интерактивная классификация доступна в `app.py`.
Сводные результаты эксперимента доступны в `reports/SUMMARY.md`.
